# Aprobación manual

La tarea continúa sólo cuando Unity Catalog contiene el tag de aprobación.

In [ ]:
from iris_mlflow_utils import build_deployment_config
from mlflow.tracking import MlflowClient

dbutils.widgets.text('model_name', 'workspace.default.iris_classifier')
dbutils.widgets.text('model_version', '')
model_name = dbutils.widgets.get('model_name')
model_version = dbutils.widgets.get('model_version')
config = build_deployment_config()
client = MlflowClient(registry_uri='databricks-uc')
version = client.get_model_version(model_name, model_version)
if version.tags.get('evaluation_status') != 'passed':
    raise RuntimeError('La versión no tiene una evaluación aprobada.')
approval = version.tags.get(config.required_approval_tag, '')
if approval != 'Approved':
    raise RuntimeError(f'Pendiente de aprobación: {config.required_approval_tag}=Approved')
client.set_model_version_tag(model_name, model_version, 'approval_status', 'approved')
dbutils.jobs.taskValues.set(key='approval_status', value='approved')
print({'model_name': model_name, 'model_version': model_version, 'approval': approval})
